In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

GPKG = "data/catastro_manzana.gpkg"
CLOUD = "data/nube_manzana_2023.laz"

GROUND_CLASS = 2
BUILDING_CLASS = 6

INNER_MARGIN = 0.5    # Meters removed from the footprint edge, so that facade points of neighbors do not count
GROUND_RING = 10      # Meters around a footprint where we look for ground points
MIN_POINTS = 50       # Below this number of roof points the height is estimated from the floors
FLOOR_HEIGHT = 3.5    # Meters per floor, only for the estimate [ESTIMADO]

In [ ]:
# Celda 2
import numpy as np
import laspy
import shapely
import geopandas as gpd

In [ ]:
# Celda 3
# Load footprints and point cloud:
parts = gpd.read_file(GPKG, layer="building_parts")
cloud = laspy.read(CLOUD)
x = np.array(cloud.x)
y = np.array(cloud.y)
z = np.array(cloud.z)
classes = np.array(cloud.classification)
print(len(parts), "building parts |", len(x), "points")

In [ ]:
# Celda 4
# Ground points that are not inside any building (a roof terrace can be classified as ground):
all_buildings = parts.geometry.union_all()
is_ground = classes == GROUND_CLASS
is_ground[is_ground] = ~shapely.contains_xy(all_buildings, x[is_ground], y[is_ground])
gx = x[is_ground]
gy = y[is_ground]
gz = z[is_ground]
print(len(gz), "ground points in streets and courtyards")

is_building = classes == BUILDING_CLASS
bx = x[is_building]
by = y[is_building]
bz = z[is_building]
print(len(bz), "building points")

In [ ]:
# Celda 5
# Helper: points of (px, py, pz) that fall inside a polygon:
def points_inside(polygon, px, py, pz):
    xmin, ymin, xmax, ymax = polygon.bounds
    near = (px >= xmin) & (px <= xmax) & (py >= ymin) & (py <= ymax)
    inside = shapely.contains_xy(polygon, px[near], py[near])
    return pz[near][inside]

In [ ]:
# Celda 6
# For each building part: ground level around it and roof heights inside it:
# z_base = 10th percentile of the ground in a ring of GROUND_RING meters (the lower side of a sloping street)
# z_p50, z_p90, z_max = percentiles of the roof points inside the footprint
z_base = []
z_p50 = []
z_p90 = []
z_max = []
n_roof = []
for polygon in parts.geometry:
    ring = polygon.buffer(GROUND_RING).difference(polygon)
    ground = points_inside(ring, gx, gy, gz)
    roof = points_inside(polygon.buffer(-INNER_MARGIN), bx, by, bz)

    if len(ground) > 0:
        z_base.append(np.percentile(ground, 10))
    else:
        z_base.append(np.nan)

    n_roof.append(len(roof))
    if len(roof) > 0:
        z_p50.append(np.percentile(roof, 50))
        z_p90.append(np.percentile(roof, 90))
        z_max.append(roof.max())
    else:
        z_p50.append(np.nan)
        z_p90.append(np.nan)
        z_max.append(np.nan)

parts["z_base"] = np.round(z_base, 2)
parts["z_p50"] = np.round(z_p50, 2)
parts["z_p90"] = np.round(z_p90, 2)
parts["z_max"] = np.round(z_max, 2)
parts["n_roof"] = n_roof

In [ ]:
# Celda 7
# Parts without ground nearby take the median ground of the block:
block_ground = float(np.median(gz))
missing = parts["z_base"].isna()
parts.loc[missing, "z_base"] = round(block_ground, 2)
print(missing.sum(), "parts without ground nearby -> z_base =", round(block_ground, 2))

In [ ]:
# Celda 8
# LOD1 height: z_p90 - z_base when there are enough roof points, otherwise floors x FLOOR_HEIGHT:
heights = []
methods = []
for i, row in parts.iterrows():
    floors = row["numberOfFloorsAboveGround"]
    if row["n_roof"] >= MIN_POINTS:
        heights.append(round(row["z_p90"] - row["z_base"], 2))
        methods.append("cloud_p90")
    else:
        heights.append(round(floors * FLOOR_HEIGHT, 2))
        methods.append("ESTIMADO_floors")

parts["height_m"] = heights
parts["height_method"] = methods
print(parts["height_method"].value_counts())

In [ ]:
# Celda 9
# Sanity check: meters per floor (a normal value is about 3 to 5):
parts["m_per_floor"] = (parts["height_m"] / parts["numberOfFloorsAboveGround"].replace(0, np.nan)).round(2)
columns = ["parcel", "numberOfFloorsAboveGround", "area_m2", "n_roof", "z_base", "z_p90", "z_max", "height_m", "m_per_floor", "height_method"]
main = ["0139402VK4703G", "0139423VK4703G", "0139424VK4703G"]
parts[parts["parcel"].isin(main)][columns].sort_values(["parcel", "height_m"])

In [ ]:
# Celda 10
# Parts that look strange: very low or very high meters per floor, or estimated:
strange = parts[(parts["m_per_floor"] < 2.5) | (parts["m_per_floor"] > 6) | (parts["height_method"] != "cloud_p90")]
print(len(strange), "parts to review")
strange[columns]

In [ ]:
# Celda 11
# Save the heights in a new layer of the same GeoPackage:
parts.to_file(GPKG, layer="building_parts_alturas", driver="GPKG")
print("Saved:", GPKG)